In [0]:
from pyspark.sql import functions as F

products_bronze = spark.table(
    "workspace.fashion_bronze.products_ingested"
)

print("Bronze product records:", products_bronze.count())
products_bronze.printSchema()

Bronze product records: 10000
root
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- color: string (nullable = true)
 |-- size: string (nullable = true)
 |-- price: double (nullable = true)
 |-- season: string (nullable = true)
 |-- product_status: string (nullable = true)



In [0]:
products_bronze.select(
    F.count("*").alias("total_products"),
    F.countDistinct("product_id").alias("unique_products"),
    F.sum(F.col("product_id").isNull().cast("int")).alias("null_product_ids"),
    F.countDistinct("category").alias("categories"),
    F.countDistinct("subcategory").alias("subcategories"),
    F.countDistinct("brand").alias("brands"),
    F.min("price").alias("minimum_price"),
    F.max("price").alias("maximum_price")
).show()

+--------------+---------------+----------------+----------+-------------+------+-------------+-------------+
|total_products|unique_products|null_product_ids|categories|subcategories|brands|minimum_price|maximum_price|
+--------------+---------------+----------------+----------+-------------+------+-------------+-------------+
|         10000|          10000|               0|         7|           31|    10|       201.07|      9982.26|
+--------------+---------------+----------------+----------+-------------+------+-------------+-------------+



In [0]:
for column in ["category", "subcategory", "brand", "color", "size", "season", "product_status"]:
    print(f"\n{column.upper()}")
    products_bronze.groupBy(column).count().orderBy(column).show(50, truncate=False)


CATEGORY
+----------------+-----+
|category        |count|
+----------------+-----+
|Accessories     |1435 |
|Ethnic Wear     |1432 |
|Footwear        |1436 |
|Kids' Clothing  |1448 |
|Men's Clothing  |1368 |
|Sportswear      |1444 |
|Women's Clothing|1437 |
+----------------+-----+


SUBCATEGORY
+---------------+-----+
|subcategory    |count|
+---------------+-----+
|Backpacks      |288  |
|Belts          |329  |
|Dresses        |524  |
|Ethnic Kurtas  |283  |
|Flats          |265  |
|Formal Shoes   |300  |
|Handbags       |271  |
|Heels          |292  |
|Hoodies        |281  |
|Jackets        |233  |
|Jeans          |721  |
|Kurtas         |249  |
|Kurtis         |269  |
|Leggings       |305  |
|Lehengas       |271  |
|Salwar Suits   |300  |
|Sandals        |296  |
|Sarees         |304  |
|Sherwanis      |274  |
|Shirts         |489  |
|Shorts         |274  |
|Skirts         |232  |
|Sneakers       |283  |
|Sports Shorts  |303  |
|Sports T-Shirts|290  |
|T-Shirts       |537  |
|Tops

In [0]:
print("Total records:", products_bronze.count())

print("Duplicate product IDs:",
      products_bronze.groupBy("product_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

for column in products_bronze.columns:
    null_count = products_bronze.filter(
        F.col(column).isNull()
    ).count()
    print(f"{column}: {null_count} null values")

Total records: 10000
Duplicate product IDs: 0
product_id: 0 null values
category: 0 null values
subcategory: 0 null values
brand: 0 null values
color: 0 null values
size: 0 null values
price: 0 null values
season: 0 null values
product_status: 0 null values


In [0]:
products_silver = (
    products_bronze
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("subcategory", F.initcap(F.trim(F.col("subcategory"))))
    .withColumn("brand", F.trim(F.col("brand")))
    .withColumn("color", F.initcap(F.trim(F.col("color"))))
    .withColumn("size", F.upper(F.trim(F.col("size"))))
    .withColumn("season", F.initcap(F.trim(F.col("season"))))
    .withColumn("product_status", F.initcap(F.trim(F.col("product_status"))))
)

In [0]:
print("Total records:", products_silver.count())

print("Duplicate product IDs:",
      products_silver.groupBy("product_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

print("Products with invalid prices:",
      products_silver.filter(F.col("price") <= 0).count())

print("Products with blank categories:",
      products_silver.filter(
          F.col("category").isNull() | (F.trim(F.col("category")) == "")
      ).count())

print("Products with blank brands:",
      products_silver.filter(
          F.col("brand").isNull() | (F.trim(F.col("brand")) == "")
      ).count())

Total records: 10000
Duplicate product IDs: 0
Products with invalid prices: 0
Products with blank categories: 0
Products with blank brands: 0


In [0]:
price_ranges = {
    "Men's Clothing": (399, 4999),
    "Women's Clothing": (399, 5999),
    "Kids' Clothing": (299, 2499),
    "Footwear": (599, 6999),
    "Accessories": (199, 4999),
    "Ethnic Wear": (799, 9999),
    "Sportswear": (499, 4999)
}

for category, (min_price, max_price) in price_ranges.items():
    invalid_count = products_silver.filter(
        (F.col("category") == category) &
        ((F.col("price") < min_price) | (F.col("price") > max_price))
    ).count()

    print(f"{category}: {invalid_count} invalid prices")

Men's Clothing: 0 invalid prices
Women's Clothing: 0 invalid prices
Kids' Clothing: 0 invalid prices
Footwear: 0 invalid prices
Accessories: 0 invalid prices
Ethnic Wear: 0 invalid prices
Sportswear: 0 invalid prices


In [0]:
products_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.fashion_silver.products")

In [0]:
products_silver_saved = spark.table(
    "workspace.fashion_silver.products"
)

print("Silver product records:", products_silver_saved.count())

products_silver_saved.printSchema()

products_silver_saved.show(5, truncate=False)

Silver product records: 10000
root
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- color: string (nullable = true)
 |-- size: string (nullable = true)
 |-- price: double (nullable = true)
 |-- season: string (nullable = true)
 |-- product_status: string (nullable = true)

+----------+----------------+-----------+-------------+------+-----+-------+----------+--------------+
|product_id|category        |subcategory|brand        |color |size |price  |season    |product_status|
+----------+----------------+-----------+-------------+------+-----+-------+----------+--------------+
|3750      |Ethnic Wear     |Sherwanis  |House of Nova|Green |XL   |8325.07|Winter    |Active        |
|3751      |Footwear        |Flats      |Vivid Vogue  |White |UK 10|5834.53|All Season|Discontinued  |
|3752      |Ethnic Wear     |Sherwanis  |Northstar    |Maroon|XS   |2020.56|Monsoon   |Activ